In [10]:
from crewai import Agent, Task, Crew,LLM
from crewai.flow.flow import Flow, start, listen
from crewai.flow.human_feedback import human_feedback, HumanFeedbackResult
from pydantic import BaseModel

import os
from dotenv import load_dotenv
load_dotenv()

True

In [11]:
llm = LLM(
    model = "groq/llama-3.3-70b-versatile"
)

In [12]:

# 🔹 Config
MAX_ITERATIONS = 3

In [ ]:
# 🔹 Research Agent (Deep, structured, factual)
research_agent = Agent(
    role="Senior Research Analyst",
    goal=(
        "Produce accurate, structured, and up-to-date research that can be directly "
        "used by a writer to create a high-quality blog."
    ),
    backstory=(
        "You are a professional research analyst with experience in analyzing complex topics, "
        "fact-checking information, and organizing insights into structured notes."
    ),
    llm=llm,
    max_tokens=300,
    verbose=True
)


# 🔹 Writer Agent (High-quality blog generator)
writer_agent = Agent(
    role="Expert Blog Writer",
    goal=(
        "Write an engaging, well-structured, and informative blog using provided research. "
        "Ensure clarity, readability, and logical flow."
    ),
    backstory=(
        "You are a professional content writer specializing in long-form blogs. "
        "You write with a clear introduction, strong body sections, and a concise conclusion. "
        "You adapt tone based on audience and incorporate feedback effectively."
    ),
    llm=llm,
    max_tokens=300,
    verbose=True
)

In [13]:

# 🔹 Structured State
class BlogState(BaseModel):
    topic: str = ""
    blog: str = ""
    feedback: str = ""
    approved: bool = False
    iteration: int = 0

In [14]:
class BlogFlow(Flow[BlogState]):

    # ── Step 1: Entry point ───────────────────
    @start()
    def get_topic(self):
        print(f"\n📝 Topic: {self.state.topic}\n")
        return self.state.topic


    # ── Step 2: Crew runs here ONLY ───────────
    @listen(get_topic)
    def generate_blog(self, topic):

        self.state.iteration += 1
        print(f"\n🔁 Iteration: {self.state.iteration}\n")

        research_task = Task(
            description=f"""
            Research the topic: {topic}
            Provide key concepts, insights, examples, trends.
            """,
            expected_output="Structured research notes",
            agent=research_agent,
        )

        writer_task = Task(
            description=f"""
            Write a high-quality blog on: {topic}
            {"Incorporate this feedback: " + self.state.feedback if self.state.feedback else ""}
            Iteration: {self.state.iteration}
            """,
            expected_output="Final blog article",
            agent=writer_agent,
        )

        crew = Crew(
            agents=[research_agent, writer_agent],
            tasks=[research_task, writer_task],
            verbose=True,
        )
        
        result = crew.kickoff(inputs = {"topic":f"{self.state.topic}"})

        self.state.blog = writer_task.output.raw
        print("\n📄 Generated Blog:\n")
        print(self.state.blog)

        return self.state.blog


    # ── Step 3: HITL happens here ONLY ────────
    @listen(generate_blog)
    @human_feedback(
        message="Review the blog. Type 'approved' or provide revision feedback:",
        emit=["approved", "needs_revision"],
        llm=llm,
        default_outcome="needs_revision",
    )
    def review_blog(self, blog):
        return self.state.blog   # just shows the blog to human


    # ── Step 4: Approved ──────────────────────
    @listen("approved")
    def on_approved(self, result: HumanFeedbackResult):
        self.state.approved = True
        print("\n✅ Blog Approved!\n")
        print(self.state.blog)


    # ── Step 5: Needs revision → loop back ────
    @listen("needs_revision")
    def on_needs_revision(self, result: HumanFeedbackResult):
        self.state.feedback = result.feedback
        print(f"\n🛠 Feedback: {self.state.feedback}\n")

        if self.state.iteration >= MAX_ITERATIONS:
            print(f"\n⚠ Max revisions reached. Saving last version.\n")
            self.state.approved = True
            return

        print("🔁 Regenerating with feedback...\n")
        return self.generate_blog(self.state.topic)


# 🔹 Run
if __name__ == "__main__":
    flow = BlogFlow()

    flow.kickoff(
        inputs={
            "topic": "How AI is Transforming Healthcare in 2025"
        }
    )

╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: BlogFlow                                                                                                 │
│  ID: c3c07ea4-2e36-4783-9655-ba88ff20ae87                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.10.1                                                                                        │
│  Latest version:  1.11.0                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: BlogFlow                                                                                                 │
│  ID: c3c07ea4-2e36-4783-9655-ba88ff20ae87                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Flow started with ID: c3c07ea4-2e36-4783-9655-ba88ff20ae87

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: get_topic                                                                                              │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: review_blog                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


✅ Blog Approved!

**How AI is Transforming Healthcare in 2025**

The healthcare sector is on the cusp of a revolution, driven by the rapid advancement and integration of Artificial Intelligence (AI). AI is transforming the way healthcare services are delivered, consumed, and experienced. The technology is improving patient outcomes, streamlining clinical workflows, and enhancing the overall quality of care. In this blog, we will delve into the current state of AI in healthcare, highlighting key concepts, insights, examples, and trends that are shaping the industry in 2025.

**The Power of Machine Learning, Deep Learning, and Natural Language Processing**

At the heart of AI in healthcare are technologies like Machine Learning (ML), Deep Learning (DL), and Natural Language Processing (NLP). ML, a subset of AI, enables systems to learn from data without being explicitly programmed. This technology is being used in healthcare for predictive analytics, disease diagnosis, and personalized 

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: on_approved                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: on_approved                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── ✅ Flow Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Execution Completed                                                                                       │
│  Name: BlogFlow                                                                                                 │
│  ID: c3c07ea4-2e36-4783-9655-ba88ff20ae87                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯